# 02 - Perfilado de la unión canónica

Objetivo: conocer el estado actual de `Canonico_Union_Trabajo.csv` antes de cualquier limpieza.

Esta libreta **no modifica la base**, no deduplica, no normaliza valores de forma permanente y no vuelve a guardar el archivo canónico. Solo genera `resumen_perfilado.csv` con indicadores de completitud.

In [1]:
import os
import re
import html
import unicodedata
import pandas as pd
from IPython.display import display

archivo = "../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv"
carpeta_salida = "../03_perfilado"

os.makedirs(carpeta_salida, exist_ok=True)

df = pd.read_csv(
    archivo,
    dtype=str,
    keep_default_na=False
)

print("Archivo cargado:", archivo)
print("Dimensiones:", df.shape)

Archivo cargado: ../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv
Dimensiones: (2153, 16)


## 1. Validación inicial

In [2]:
columnas_esperadas = [
    "Base_origen",
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "SubArea",
    "Keywords",
    "Abstract"
]

conteos_esperados = {
    "ISBD": 245,
    "CC": 122,
    "IA": 710,
    "RS": 178,
    "TC": 580,
    "SIAV": 318
}

if df.shape != (2153, 16):
    raise ValueError(f"Dimensiones inesperadas: {df.shape}")

if list(df.columns) != columnas_esperadas:
    raise ValueError("Las columnas o su orden no coinciden con la estructura esperada.")

if set(df["Base_origen"].unique()) != set(conteos_esperados):
    raise ValueError("Base_origen contiene etiquetas inesperadas.")

conteos_base = df["Base_origen"].value_counts().to_dict()

if conteos_base != conteos_esperados:
    raise ValueError(f"Los conteos por Base_origen no coinciden: {conteos_base}")

print("Validación inicial: OK")
print("Filas:", len(df))
print("Columnas:", len(df.columns))

Validación inicial: OK
Filas: 2153
Columnas: 16


## 2. Estructura general

In [3]:
print("Columnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
display(df.dtypes.rename("tipo").to_frame())

print("\nFilas por Base_origen:")
display(df["Base_origen"].value_counts().rename_axis("Base_origen").reset_index(name="Filas"))

print("\nFilas por Fuente_origen:")
display(df["Fuente_origen"].value_counts().rename_axis("Fuente_origen").reset_index(name="Filas"))

Columnas:
['Base_origen', 'Fuente_origen', 'indice', 'Titulo', 'Año', 'Autor_norm', 'Afiliacion1', 'Afiliacion2', 'ISBN', 'ISSN', 'Doi', 'URL', 'Area', 'SubArea', 'Keywords', 'Abstract']

Tipos de datos:


,tipo
Base_origen,object
Fuente_origen,object
indice,object
Titulo,object
Año,object
Autor_norm,object
Afiliacion1,object
Afiliacion2,object
ISBN,object
ISSN,object



Filas por Base_origen:


,Base_origen,Filas
0,IA,710
1,TC,580
2,SIAV,318
3,ISBD,245
4,RS,178
5,CC,122



Filas por Fuente_origen:


,Fuente_origen,Filas
0,Scopus,1285
1,IEEE,399
2,EV,253
3,WoS,85
4,ACM,66
5,ScienceDirect,51
6,ProQuest,14


## 3. Completitud y valores únicos

In [4]:
filas_resumen = []

for columna in df.columns:
    vacios = df[columna].astype(str).str.strip().eq("")
    no_vacios = ~vacios

    filas_resumen.append({
        "Columna": columna,
        "Total": len(df),
        "No_vacios": int(no_vacios.sum()),
        "Vacios": int(vacios.sum()),
        "Completitud": round(no_vacios.mean() * 100, 2),
        "Unicos": int(df.loc[no_vacios, columna].nunique())
    })

perfil = pd.DataFrame(filas_resumen)
perfil = perfil.sort_values("Completitud").reset_index(drop=True)

display(perfil)

resumen_perfilado = perfil[
    ["Columna", "Total", "Vacios", "Completitud", "Unicos"]
].copy()

resumen_perfilado.to_csv(
    f"{carpeta_salida}/resumen_perfilado.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Completitud promedio:", round(perfil["Completitud"].mean(), 2), "%")
print("Resumen guardado en:", f"{carpeta_salida}/resumen_perfilado.csv")

,Columna,Total,No_vacios,Vacios,Completitud,Unicos
0,SubArea,2153,0,2153,0.00,0
1,ISBN,2153,1061,1092,49.28,216
2,Afiliacion2,2153,1386,767,64.38,401
3,ISSN,2153,1592,561,73.94,264
4,URL,2153,1900,253,88.25,549
5,Año,2153,2028,125,94.19,4
6,Doi,2153,2080,73,96.61,446
7,Keywords,2153,2139,14,99.35,667
8,Abstract,2153,2151,2,99.91,629
9,Autor_norm,2153,2153,0,100.00,673


Completitud promedio: 85.37 %
Resumen guardado en: ../03_perfilado/resumen_perfilado.csv


## 4. Perfilado de `Autor_norm`

In [5]:
# Para contar autores se trabaja con una copia temporal.
# html.unescape evita romper entidades HTML que puedan contener ';'.

autor_temporal = df["Autor_norm"].map(html.unescape)

def contar_autores(valor):
    if not valor.strip():
        return 0
    return len([parte for parte in valor.split(";") if parte.strip()])

def tiene_forma_abreviada(valor):
    partes = [p.strip() for p in valor.split(";") if p.strip()]

    for parte in partes:
        # Quitar temporalmente el contenido final entre paréntesis para revisar el nombre.
        nombre = re.sub(r"\s*\([^)]*\)\s*$", "", parte).strip()

        if re.search(r"^(?:[A-ZÁÉÍÓÚÑ]\.\s*){2,}", nombre):
            return True

        if re.search(r",\s*(?:[A-ZÁÉÍÓÚÑ]\.?\s*){1,5}$", nombre):
            return True

    return False

cantidad_autores = autor_temporal.map(contar_autores)

patron_numerado = df["Autor_norm"].str.contains(
    r"\(\s*\d+(?:\s*,\s*\d+)*\s*\)", regex=True
)

posibles_ids_scopus = df["Autor_norm"].str.contains(
    r"\(\s*\d{7,12}\s*\)", regex=True
)

entidades_html = df["Autor_norm"].str.contains(
    r"&#(?:x[0-9A-Fa-f]+|\d+);", regex=True
)

formas_abreviadas = autor_temporal.map(tiene_forma_abreviada)

separadores_sospechosos = autor_temporal.str.contains(
    r"(?:^\s*;|;\s*$|;;)", regex=True
)

print("Celdas no vacías:", int(df["Autor_norm"].str.strip().ne("").sum()))
print("Valores completos distintos de Autor_norm:", df["Autor_norm"].nunique())
print("Filas con un solo autor:", int((cantidad_autores == 1).sum()))
print("Filas con varios autores:", int((cantidad_autores > 1).sum()))
print("Máximo aproximado de autores en una fila:", int(cantidad_autores.max()))
print("Filas con patrones numerados como (1), (2), etc.:", int(patron_numerado.sum()))
print("Filas con posibles IDs largos de Scopus:", int(posibles_ids_scopus.sum()))
print("Filas con entidades HTML &#...;:", int(entidades_html.sum()))
print("Filas con formas muy abreviadas mediante iniciales:", int(formas_abreviadas.sum()))
print("Filas con separadores ';' obviamente mal formados:", int(separadores_sospechosos.sum()))

print("\nFuente_origen de las filas con posibles IDs de Scopus:")
display(
    df.loc[posibles_ids_scopus, "Fuente_origen"]
      .value_counts()
      .rename_axis("Fuente_origen")
      .reset_index(name="Filas")
)

Celdas no vacías: 2153
Valores completos distintos de Autor_norm: 673
Filas con un solo autor: 172
Filas con varios autores: 1981
Máximo aproximado de autores en una fila: 151
Filas con patrones numerados como (1), (2), etc.: 1538
Filas con posibles IDs largos de Scopus: 1285
Filas con entidades HTML &#...;: 0
Filas con formas muy abreviadas mediante iniciales: 558
Filas con separadores ';' obviamente mal formados: 0

Fuente_origen de las filas con posibles IDs de Scopus:


,Fuente_origen,Filas
0,Scopus,1285


## 5. Perfilado de afiliaciones

In [6]:
def texto_sin_acentos(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower()

af1_temp = df["Afiliacion1"].map(texto_sin_acentos)
af2_temp = df["Afiliacion2"].map(texto_sin_acentos)

patron_unam = (
    r"\bunam\b|"
    r"u\.?\s*n\.?\s*a\.?\s*m\.?|"
    r"universidad nacional autonoma de mexico|"
    r"national autonomous university of mexico"
)

unam_af1 = af1_temp.str.contains(patron_unam, regex=True)
unam_af2 = af2_temp.str.contains(patron_unam, regex=True)

print("Afiliacion1 vacía:", int(df["Afiliacion1"].str.strip().eq("").sum()))
print("Afiliacion1 valores únicos:", df.loc[df["Afiliacion1"].str.strip().ne(""), "Afiliacion1"].nunique())
print("Afiliacion2 vacía:", int(df["Afiliacion2"].str.strip().eq("").sum()))
print("Afiliacion2 valores únicos:", df.loc[df["Afiliacion2"].str.strip().ne(""), "Afiliacion2"].nunique())

print("\nFilas con UNAM en Afiliacion1:", int(unam_af1.sum()))
print("Filas con UNAM en Afiliacion2:", int(unam_af2.sum()))
print("Filas con UNAM en alguna afiliación:", int((unam_af1 | unam_af2).sum()))
print("Filas con UNAM en ambas afiliaciones:", int((unam_af1 & unam_af2).sum()))
print("Filas sin UNAM en ambas afiliaciones:", int((~(unam_af1 | unam_af2)).sum()))

print("\nFilas con patrones numerados en Afiliacion1:",
      int(df["Afiliacion1"].str.contains(r"\(\d+\)", regex=True).sum()))
print("Filas con patrones numerados en Afiliacion2:",
      int(df["Afiliacion2"].str.contains(r"\(\d+\)", regex=True).sum()))

Afiliacion1 vacía: 0
Afiliacion1 valores únicos: 650
Afiliacion2 vacía: 767
Afiliacion2 valores únicos: 401

Filas con UNAM en Afiliacion1: 2131
Filas con UNAM en Afiliacion2: 1337
Filas con UNAM en alguna afiliación: 2147
Filas con UNAM en ambas afiliaciones: 1321
Filas sin UNAM en ambas afiliaciones: 6

Filas con patrones numerados en Afiliacion1: 253
Filas con patrones numerados en Afiliacion2: 0


## 6. Perfilado de DOI

In [7]:
def normalizar_doi(valor):
    doi = str(valor).strip().lower()
    doi = re.sub(r"^doi\s*:\s*", "", doi)
    doi = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", doi)
    doi = re.sub(r"\s+", "", doi)
    return doi

doi_temp = df["Doi"].map(normalizar_doi)
doi_presente = doi_temp.ne("")

conteos_doi = doi_temp[doi_presente].value_counts()
doi_repetidos = conteos_doi[conteos_doi > 1]

doi_grupos = (
    df.loc[doi_presente, ["Base_origen", "Fuente_origen"]]
      .assign(doi_temp=doi_temp[doi_presente])
      .groupby("doi_temp")
)

doi_varias_bases = doi_grupos["Base_origen"].nunique() > 1
doi_varias_fuentes = doi_grupos["Fuente_origen"].nunique() > 1

print("DOI no vacíos:", int(doi_presente.sum()))
print("DOI vacíos:", int((~doi_presente).sum()))
print("DOI únicos después de normalización temporal:", int(doi_temp[doi_presente].nunique()))
print("DOI repetidos (grupos):", int(len(doi_repetidos)))
print("Filas involucradas en DOI repetidos:", int(doi_repetidos.sum()))
print("DOI presentes en más de una Base_origen:", int(doi_varias_bases.sum()))
print("DOI presentes en más de una Fuente_origen:", int(doi_varias_fuentes.sum()))

DOI no vacíos: 2080
DOI vacíos: 73
DOI únicos después de normalización temporal: 445
DOI repetidos (grupos): 418
Filas involucradas en DOI repetidos: 2053
DOI presentes en más de una Base_origen: 229
DOI presentes en más de una Fuente_origen: 155


## 7. Perfilado de títulos

In [8]:
def normalizar_titulo(valor):
    texto = unicodedata.normalize("NFKC", str(valor)).casefold().strip()
    texto = re.sub(r"[^\w\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

titulo_temp = df["Titulo"].map(normalizar_titulo)

# Solo para comparación bibliográfica, no modifica Año.
anio_temp = df["Año"].str.strip().str.replace(
    r"^(\d{4})\.0$", r"\1", regex=True
)

conteos_titulo = titulo_temp[titulo_temp.ne("")].value_counts()
titulos_repetidos = conteos_titulo[conteos_titulo > 1]

conteos_titulo_anio = (
    pd.DataFrame({
        "Titulo_temp": titulo_temp,
        "Año_temp": anio_temp
    })
    .value_counts()
)

titulo_anio_repetidos = conteos_titulo_anio[conteos_titulo_anio > 1]

print("Títulos no vacíos:", int(titulo_temp.ne("").sum()))
print("Títulos únicos originales:", df["Titulo"].nunique())
print("Títulos únicos normalizados temporalmente:", int(titulo_temp.nunique()))
print("Títulos repetidos normalizados (grupos):", int(len(titulos_repetidos)))
print("Filas involucradas en títulos repetidos:", int(titulos_repetidos.sum()))
print("Titulo + Año repetidos (grupos):", int(len(titulo_anio_repetidos)))
print("Filas involucradas en Titulo + Año repetidos:", int(titulo_anio_repetidos.sum()))

Títulos no vacíos: 2153
Títulos únicos originales: 482
Títulos únicos normalizados temporalmente: 462
Títulos repetidos normalizados (grupos): 436
Filas involucradas en títulos repetidos: 2127
Titulo + Año repetidos (grupos): 466
Filas involucradas en Titulo + Año repetidos: 2109


## 8. Otros identificadores y metadatos

In [9]:
print("Valores y frecuencias de Año:")
display(
    df["Año"].replace("", "[VACIO]")
      .value_counts(dropna=False)
      .rename_axis("Año")
      .reset_index(name="Filas")
)

anio_no_vacio = df["Año"].str.strip().ne("")
anio_formato_anomalo = anio_no_vacio & ~df["Año"].str.strip().str.fullmatch(r"\d{4}")

print("Año vacío:", int((~anio_no_vacio).sum()))
print("Año con formato distinto de cuatro dígitos:", int(anio_formato_anomalo.sum()))

print("\nCampos seleccionados:")
for columna in ["ISBN", "ISSN", "URL", "Keywords", "Abstract"]:
    vacios = int(df[columna].str.strip().eq("").sum())
    no_vacios = len(df) - vacios
    completitud = round(no_vacios / len(df) * 100, 2)
    unicos = df.loc[df[columna].str.strip().ne(""), columna].nunique()

    print(
        f"{columna}: vacíos={vacios}, no vacíos={no_vacios}, "
        f"completitud={completitud}%, únicos={unicos}"
    )

isbn_cientifico = df["ISBN"].str.strip().str.fullmatch(
    r"[+-]?\d+(?:\.\d+)?[Ee][+-]?\d+"
)
issn_cientifico = df["ISSN"].str.strip().str.fullmatch(
    r"[+-]?\d+(?:\.\d+)?[Ee][+-]?\d+"
)

print("\nISBN con apariencia de notación científica:", int(isbn_cientifico.sum()))
print("ISSN con apariencia de notación científica:", int(issn_cientifico.sum()))

Valores y frecuencias de Año:


,Año,Filas
0,2025,1018
1,2024,882
2,[VACIO],125
3,2025.0,73
4,2024.0,55


Año vacío: 125
Año con formato distinto de cuatro dígitos: 128

Campos seleccionados:
ISBN: vacíos=1092, no vacíos=1061, completitud=49.28%, únicos=216
ISSN: vacíos=561, no vacíos=1592, completitud=73.94%, únicos=264
URL: vacíos=253, no vacíos=1900, completitud=88.25%, únicos=549
Keywords: vacíos=14, no vacíos=2139, completitud=99.35%, únicos=667
Abstract: vacíos=2, no vacíos=2151, completitud=99.91%, únicos=629

ISBN con apariencia de notación científica: 0
ISSN con apariencia de notación científica: 0


## 9. `Area` y `SubArea`

In [10]:
print("Frecuencias de Area:")
display(
    df["Area"].replace("", "[VACIO]")
      .value_counts()
      .rename_axis("Area")
      .reset_index(name="Filas")
)

print("\nFrecuencias de SubArea:")
display(
    df["SubArea"].replace("", "[VACIO]")
      .value_counts()
      .rename_axis("SubArea")
      .reset_index(name="Filas")
)

print("Area vacía:", int(df["Area"].str.strip().eq("").sum()))
print("SubArea vacía:", int(df["SubArea"].str.strip().eq("").sum()))
print("Valores no vacíos únicos de Area:",
      df.loc[df["Area"].str.strip().ne(""), "Area"].nunique())
print("Valores no vacíos únicos de SubArea:",
      df.loc[df["SubArea"].str.strip().ne(""), "SubArea"].nunique())

Frecuencias de Area:


,Area,Filas
0,IA,644
1,SIAV,568
2,CC,368
3,TC,213
4,ISBD,185
5,RS,175



Frecuencias de SubArea:


,SubArea,Filas
0,[VACIO],2153


Area vacía: 0
SubArea vacía: 2153
Valores no vacíos únicos de Area: 6
Valores no vacíos únicos de SubArea: 0


## 10. Repetición aproximada entre bases

In [11]:
# Clave temporal:
# 1) DOI normalizado, si existe.
# 2) Si no hay DOI, Titulo normalizado + Año temporal.
# No se guarda esta clave en la base.

clave_temporal = []

for doi, titulo, anio in zip(doi_temp, titulo_temp, anio_temp):
    if doi:
        clave_temporal.append("DOI:" + doi)
    elif titulo:
        clave_temporal.append("TY:" + titulo + "|" + anio)
    else:
        clave_temporal.append("")

clave_temporal = pd.Series(clave_temporal, index=df.index)

grupos_bibliograficos = (
    df.assign(Clave_temporal=clave_temporal)
      .loc[clave_temporal.ne("")]
      .groupby("Clave_temporal")
)

tamano_grupos = grupos_bibliograficos.size()
grupos_varias_bases = grupos_bibliograficos["Base_origen"].nunique() > 1
grupos_varias_fuentes = grupos_bibliograficos["Fuente_origen"].nunique() > 1

print("Grupos bibliográficos aproximados:", int(len(tamano_grupos)))
print("Grupos con más de una fila:", int((tamano_grupos > 1).sum()))
print("Filas involucradas en grupos con más de una fila:",
      int(tamano_grupos[tamano_grupos > 1].sum()))
print("Grupos presentes en más de una Base_origen:",
      int(grupos_varias_bases.sum()))
print("Filas involucradas en grupos de varias Base_origen:",
      int(tamano_grupos[grupos_varias_bases].sum()))
print("Grupos presentes en más de una Fuente_origen:",
      int(grupos_varias_fuentes.sum()))
print("Filas involucradas en grupos de varias Fuente_origen:",
      int(tamano_grupos[grupos_varias_fuentes].sum()))

print("\nNota: estos grupos son solo candidatos diagnósticos; no implican deduplicación.")

Grupos bibliográficos aproximados: 464
Grupos con más de una fila: 436
Filas involucradas en grupos con más de una fila: 2125
Grupos presentes en más de una Base_origen: 236
Filas involucradas en grupos de varias Base_origen: 1609
Grupos presentes en más de una Fuente_origen: 160
Filas involucradas en grupos de varias Fuente_origen: 1233

Nota: estos grupos son solo candidatos diagnósticos; no implican deduplicación.


## 11. Resumen final

In [12]:
completitud_promedio = round(perfil["Completitud"].mean(), 2)

print("Total filas:", len(df))
print("Total columnas:", len(df.columns))
print()
print("Completitud promedio:", f"{completitud_promedio}%")
print()
print("Filas con varios autores:", int((cantidad_autores > 1).sum()))
print("Filas con posible afiliación UNAM:", int((unam_af1 | unam_af2).sum()))
print()
print("DOI vacíos:", int((~doi_presente).sum()))
print("DOI únicos:", int(doi_temp[doi_presente].nunique()))
print("DOI repetidos:", int(len(doi_repetidos)), "grupos")
print()
print("Títulos repetidos:", int(len(titulos_repetidos)), "grupos")
print()
print("Filas sin ISBN:", int(df["ISBN"].str.strip().eq("").sum()))
print("Filas sin ISSN:", int(df["ISSN"].str.strip().eq("").sum()))
print("Filas sin URL:", int(df["URL"].str.strip().eq("").sum()))
print("Filas sin Keywords:", int(df["Keywords"].str.strip().eq("").sum()))
print("Filas sin Abstract:", int(df["Abstract"].str.strip().eq("").sum()))
print()
print("Area vacía:", int(df["Area"].str.strip().eq("").sum()))
print("SubArea vacía:", int(df["SubArea"].str.strip().eq("").sum()))
print()
print("Grupos bibliográficos aproximados:", int(len(tamano_grupos)))
print("Grupos presentes en varias Base_origen:", int(grupos_varias_bases.sum()))
print()
print("Perfilado terminado. La base original no fue modificada.")

Total filas: 2153
Total columnas: 16

Completitud promedio: 85.37%

Filas con varios autores: 1981
Filas con posible afiliación UNAM: 2147

DOI vacíos: 73
DOI únicos: 445
DOI repetidos: 418 grupos

Títulos repetidos: 436 grupos

Filas sin ISBN: 1092
Filas sin ISSN: 561
Filas sin URL: 253
Filas sin Keywords: 14
Filas sin Abstract: 2

Area vacía: 0
SubArea vacía: 2153

Grupos bibliográficos aproximados: 464
Grupos presentes en varias Base_origen: 236

Perfilado terminado. La base original no fue modificada.
